# EasyRAG Knowledge Graph

This notebook focuses on automatic knowledge-graph enrichment during ingestion. The goal is not to teach graph CRUD first. The goal is to show how entities and relations can be extracted from a small corpus, synchronized into retrieval state, and then surfaced in graph-aware query results.

## What you will do

- define deterministic embedding, query, and KG-extraction stubs
- index a small architecture corpus with automatic KG extraction enabled
- inspect workspace stats and graph artifacts
- run a graph-aware query and inspect citations, entities, and relations

## Why this notebook uses a KG stub

Automatic KG extraction is easiest to understand when it is reproducible. The stubbed `llm_model_func` in this notebook lets us demonstrate the full flow without depending on an external provider.


## Step 1: Import the public API and runtime helper

We use the same public API surface as the other fundamentals notebooks, plus `KGExtractionConfig` to make the extraction behavior explicit.


In [ ]:
import json
import tempfile

from easyrag import EasyRAG, KGExtractionConfig, QueryParam
from easyrag.support.async_utils import run_sync


This cell only imports the API surface we need. The next cell defines deterministic helpers so the notebook remains zero-key runnable.


## Step 2: Define deterministic embedding, query, and KG stubs

The embedding and query stubs keep retrieval stable. The KG stub simulates what an extraction model might return for a small architecture document containing an orchestrator, a workflow, and an embedding layer.


In [ ]:
_KEYWORDS = [
    "easyrag",
    "retrieval",
    "workflow",
    "embedding",
    "query",
    "rewrite",
    "entity",
    "relation",
    "component",
    "dependency",
    "service",
    "module",
]


def _stub_embedding(texts: list[str]) -> list[list[float]]:
    vectors = []
    for text in texts:
        lowered = text.lower()
        vector = [float(lowered.count(keyword)) for keyword in _KEYWORDS]
        vector.append(float(len(lowered.split())))
        vectors.append(vector)
    return vectors


def _stub_query_model(prompt: str, *, task: str, count: int = 1) -> str | list[str]:
    cleaned = prompt.split(":", 1)[-1].strip()
    if task == "rewrite":
        return cleaned
    if task == "mqe":
        return [f"{cleaned} variant {index}" for index in range(1, count + 1)]
    raise ValueError(task)


def _stub_kg_model(text: str, *, entity_types, max_entities: int, max_relations: int, metadata=None):
    del entity_types, metadata
    entities = [
        {"name": "EasyRAG", "type": "component", "description": "Repository knowledge system root."},
        {"name": "Retrieval Workflow", "type": "workflow", "description": "Retrieval flow for grounded answers."},
        {"name": "Embedding Layer", "type": "dependency", "description": "Dense retrieval support layer."},
    ]
    relations = [
        {"source": "EasyRAG", "target": "Retrieval Workflow", "relation": "orchestrates", "description": "EasyRAG orchestrates the retrieval workflow."},
        {"source": "EasyRAG", "target": "Embedding Layer", "relation": "depends_on", "description": "EasyRAG depends on embeddings for dense retrieval."},
    ]
    return {
        "entities": entities[:max_entities],
        "relations": relations[:max_relations],
    }


This cell defines the helper behavior only. There should be no runtime output yet.


## Step 3: Create a tiny architecture corpus and KG configuration

The corpus is intentionally small so the relationship between text, extracted entities, and extracted relations remains obvious. `KGExtractionConfig` makes the allowed entity types explicit for the reader.


In [ ]:
demo_texts = [
    "# Architecture\nEasyRAG uses a retrieval workflow and an embedding layer for grounded search.\n",
    "# Notes\nThe workflow coordinates retrieval while the embedding layer supports dense matching.\n",
]

demo_ids = ["doc::architecture", "doc::notes"]
demo_paths = ["docs/architecture.md", "docs/notes.md"]
kg_config = KGExtractionConfig(entity_types=("component", "workflow", "dependency"))

print(json.dumps({"ids": demo_ids, "paths": demo_paths, "entity_types": list(kg_config.entity_types)}, indent=2))


You should see the document IDs, logical paths, and the constrained entity-type allowlist. That allowlist helps make the extracted graph easier to reason about in a teaching notebook.


## Step 4: Create and initialize a graph-enabled workspace

The workspace uses deterministic helper functions and the explicit KG configuration from the previous step. This keeps the extraction reproducible while still exercising the normal EasyRAG ingestion path.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
working_dir = temp_dir.name
workspace = "knowledge-graph"

rag = EasyRAG(
    working_dir=working_dir,
    workspace=workspace,
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    llm_model_func=_stub_kg_model,
    kg_extraction_config=kg_config,
)

run_sync(rag.initialize_storages())
print(json.dumps({"working_dir": working_dir, "workspace": workspace}, indent=2))


The workspace is now ready to receive documents. The next step is where automatic KG extraction becomes visible.


## Step 5: Insert the corpus with automatic KG extraction enabled

The insertion call is the same high-level API used elsewhere, but this time the configured `llm_model_func` causes ingestion to produce entities and relations alongside chunks and summaries.


In [ ]:
insert_stats = run_sync(rag.ainsert(demo_texts, ids=demo_ids, file_paths=demo_paths))
print(json.dumps(insert_stats, indent=2))


A successful output should show nonzero entity and relation counts. That is the first sign that graph-enriched ingestion has happened.


## Step 6: Inspect aggregate workspace stats

`rag.get_stats()` gives a compact system view of what the ingestion step created. For a KG-focused walkthrough, the most interesting fields are `entity_vectors`, `relation_vectors`, `graph_nodes`, and `graph_edges`.


In [ ]:
stats = run_sync(rag.get_stats())
print(json.dumps(stats, indent=2))


You should see nonzero graph and relation counts. This shows that the graph layer is not an abstract concept here; it has already produced concrete retrieval state.


## Step 7: Inspect a couple of graph artifacts directly

It helps to look at the graph layer itself before running a query. We inspect one entity node and the relation records attached to it.


In [ ]:
easyrag_entity = run_sync(rag.graph_storage.get_node("entity::easyrag"))
easyrag_relations = run_sync(rag.graph_storage.list_relations(entity_id="entity::easyrag"))

print(json.dumps({
    "entity": easyrag_entity,
    "relations": easyrag_relations,
}, indent=2))


You should now see the extracted `EasyRAG` entity and one or more relation records that connect it to the workflow or embedding layer. This is the most direct way to verify that the graph layer contains real content before retrieval starts.


## Step 8: Run a graph-aware query and inspect the result

Now we ask a question that should benefit from graph signals. The returned result still stays grounded in citations, but it also exposes surfaced entities and relation hits that help explain the retrieval path.


In [ ]:
result = run_sync(
    rag.aquery(
        "What does EasyRAG orchestrate?",
        QueryParam(mode="local", rewrite_enabled=False, mqe_enabled=False),
    )
)

print(json.dumps({
    "citations": result.citations,
    "entities": result.entities,
    "relations": result.relations,
    "metadata": result.metadata,
}, indent=2))


The key thing to observe is the combination of outputs: citations still carry grounded snippets, while entities and relations make the graph-aware retrieval path more legible. The graph layer helps retrieval; it does not replace grounded evidence.


## Step 9: Clean up the workspace

We close the storages and remove the temporary directory so the notebook leaves no persistent state behind.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()

print("Closed storages and removed the temporary knowledge-graph workspace.")


After cleanup, the automatic KG walkthrough is complete. The main lesson is that graph extraction happens during ingestion and then becomes one more retrieval aid alongside chunks and summaries.

## Next steps

- read [docs/04-retrieval-overview.md](../../docs/04-retrieval-overview.md) for the retrieval-side architectural view
- compare this notebook with [02_query_modes.ipynb](02_query_modes.ipynb) to see how graph-aware retrieval fits into the wider mode story
- if you later want manual graph maintenance, inspect the graph operation tests and orchestrator methods such as `acreate_entity()` and `ainsert_custom_kg()`
